In [25]:
import pandas as pd

full_dataset = pd.read_csv("features.csv")

print(full_dataset.head())

train_dataset = full_dataset.sample(frac=0.8, random_state=42)
test_dataset = full_dataset.drop(train_dataset.index)

   packet_size       ttl  protocol  src_port  dst_port  flags  tcp_window  \
0     0.044118  0.196850  0.352941  0.007263  0.799482   0.96    0.001175   
1     0.003443  0.196850  0.352941  0.007263  0.799482   1.00    0.001175   
2     0.001722  0.251969  0.352941  0.799482  0.007263   0.64    0.001236   
3     0.003443  0.251969  0.352941  0.799482  0.007263   0.96    0.001236   
4     0.001722  0.251969  0.352941  0.799482  0.007263   0.68    0.001236   

   payload_size  
0      0.042457  
1      0.001724  
2      0.000000  
3      0.001724  
4      0.000000  


In [26]:
# Write a autoencoder net with binary intermediate layers

import torch
import torch.nn as nn


class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        out = torch.sign(x)
        out[out == 0] = 1
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        mask = (x.abs() <= 1).float()
        return grad_output * mask


def binarize(x):
    return BinarizeSTE.apply(x)


class BinaryHiddenLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return binarize(self.linear(x))


class BinaryAutoencoder(nn.Module):
    def __init__(self, input_size=8, hidden_size=16, latent_size=4):
        super().__init__()

        self.encoder = nn.Sequential(
            BinaryHiddenLayer(input_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, latent_size),
        )

        self.decoder_hidden = nn.Sequential(
            BinaryHiddenLayer(latent_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
            BinaryHiddenLayer(hidden_size, hidden_size),
        )
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        latent = self.encoder(x)
        hidden = self.decoder_hidden(latent)
        reconstruction = self.output_layer(hidden)
        return reconstruction, latent


In [ ]:
import numpy as np

x = torch.tensor(train_dataset.to_numpy(dtype=np.float32))

model = BinaryAutoencoder(input_size=x.shape[1], hidden_size=16, latent_size=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 2000
for epoch in range(epochs):
    optimizer.zero_grad()
    reconstruction, latent = model(x)
    loss = torch.nn.functional.mse_loss(reconstruction, x)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:5d}  loss {loss.item():.6f}")


x_test = torch.tensor(test_dataset.to_numpy(dtype=np.float32))
with torch.no_grad():
    reconstruction, latent = model(x_test)
    print("\nbinary latent codes (first 5 rows):")
    print(latent[:5])
    print("\nreconstruction error (first 5 rows):")
    print((reconstruction - x_test).abs()[:5])
    # for r in reconstr


epoch     0  loss 0.312944
epoch   200  loss 0.109489
epoch   400  loss 0.056705
epoch   600  loss 0.032784
epoch   800  loss 0.029311
epoch  1000  loss 0.032907
epoch  1200  loss 0.025606
epoch  1400  loss 0.021782
epoch  1600  loss 0.020122
epoch  1800  loss 0.023196
epoch  1999  loss 0.019264

binary latent codes (first 5 rows):
tensor([[-1., -1., -1., -1.],
        [ 1.,  1.,  1., -1.],
        [-1., -1., -1.,  1.],
        [ 1., -1.,  1.,  1.],
        [-1., -1., -1.,  1.]])

reconstruction error (first 5 rows):
tensor([[9.6677e-03, 1.3260e-01, 1.8658e-01, 1.1020e-01, 1.1375e-02, 2.9532e-01,
         2.9134e-04, 1.0262e-02],
        [6.1222e-02, 4.1323e-01, 2.7197e-01, 8.6925e-04, 5.5436e-02, 8.7808e-03,
         2.5213e-03, 5.8538e-02],
        [1.4796e-02, 2.0077e-01, 8.9359e-01, 1.7030e-01, 1.6812e-02, 4.6634e-01,
         8.1223e-03, 9.7388e-03],
        [1.9559e-03, 3.0625e-01, 9.0248e-01, 1.2595e-01, 7.6943e-01, 3.0109e-02,
         2.5027e-02, 3.0399e-02],
        [8.6925e-